# Ethereum Transaction Schema Validation v3

## tl;dr

This notebook validates FPG Digital Asset schema gaps using Ethereum transaction-level data as the main dataset. SNAP Bitcoin OTC is retained only as supporting network evidence.

## Context & Methods

The sample was extracted via Ethereum JSON-RPC from recent mainnet blocks. It includes transaction hash, from/to address, value, block timestamp, block number, nonce, transaction index, gas fields, input size, type, receipt status, and receipt gas used.

### Key Assumptions

The sample is recent-block transaction evidence, not a full-chain market distribution. Synthetic v3 policy outcomes are scenario tests, not observed fraud or regulatory failure rates.

In [ ]:
# ruff: noqa: E501, E701, E702, I001

from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd().resolve()
if ROOT.name != 'ADP-DA': ROOT = next(p for p in [ROOT, *ROOT.parents] if p.name == 'ADP-DA')
eth = pd.read_csv(ROOT/'03_digital_asset/data/raw/transactions/ethereum_transactions_sample.csv')
profile = json.loads((ROOT/'03_digital_asset/data/processed/ethereum_transaction_profile_v3.json').read_text(encoding='utf-8'))
mapping = pd.read_csv(ROOT/'03_digital_asset/data/processed/regulatory_transaction_field_mapping_v3.csv')
synthetic = pd.read_csv(ROOT/'03_digital_asset/data/processed/synthetic_regulated_transfer_v3.csv')
sensitivity = pd.DataFrame(json.loads((ROOT/'03_digital_asset/artifacts/candidate_policy_v3/policy_sensitivity_results.json').read_text(encoding='utf-8')))
controls = pd.DataFrame(json.loads((ROOT/'03_digital_asset/artifacts/candidate_policy_v3/control_coverage.json').read_text(encoding='utf-8')))
print({'ethereum_rows': len(eth), 'synthetic_v3_rows': len(synthetic), 'required_fields': len(mapping)})

## Data

### 1. Schema Profile

Check row count, columns, dtypes, missingness, hash uniqueness, duplicate transactions, and field availability.

In [ ]:
display(eth.head())
display(pd.DataFrame({'column': eth.columns, 'dtype': [str(t) for t in eth.dtypes], 'missing_rate': eth.isna().mean().round(4).values}))
print({k: profile[k] for k in ['row_count','unique_transaction_hash','duplicate_transaction_count','hash_uniqueness_ratio','from_to_availability','value_availability_ratio','timestamp_availability_ratio','status_availability_ratio']})

## Results

### 2. ETH Value Distribution

Use log and percentile views; no IQR outlier deletion is applied because blockchain values are heavy-tailed by design.

In [ ]:
value = pd.to_numeric(eth['value_eth'], errors='coerce').fillna(0)
display(pd.Series(profile['value_distribution_eth']).to_frame('value'))
value.plot.hist(bins=60, color='#2f6f8f', title='ETH value distribution')
plt.tight_layout()

### 3. log(value) Distribution

Zero-value transactions are excluded from the log chart only for numerical plotting.

In [ ]:
positive = value[value > 0]
positive.apply(lambda x: __import__('math').log10(x)).plot.hist(bins=60, color='#4f8f65', title='log10(ETH value), positive transactions only')
plt.tight_layout()

### 4. Daily Transaction Frequency

The sampling window is bounded by the fetched recent block range.

In [ ]:
eth['day'] = pd.to_datetime(eth['block_timestamp']).dt.date
daily = eth.groupby('day').size().reset_index(name='transactions')
display(daily)
daily.plot(x='day', y='transactions', legend=False, color='#446fb3', title='Daily transaction frequency in sample')
plt.xticks(rotation=45, ha='right'); plt.tight_layout()

### 5. Sender Concentration

In [ ]:
sender_top = eth['from_address'].value_counts().head(15).reset_index(); sender_top.columns=['from_address','transactions']
display(sender_top)
sender_top.plot.bar(x='from_address', y='transactions', legend=False, color='#7a6f45', title='Top sender concentration')
plt.xticks([]); plt.tight_layout()
print(profile['sender_concentration'])

### 6. Receiver Concentration

In [ ]:
receiver_top = eth['to_address'].value_counts().head(15).reset_index(); receiver_top.columns=['to_address','transactions']
display(receiver_top)
receiver_top.plot.bar(x='to_address', y='transactions', legend=False, color='#8b5a5a', title='Top receiver concentration')
plt.xticks([]); plt.tight_layout()
print(profile['receiver_concentration'])

### 7. Transaction Identity / Replay-Relevant Fields

Hash, nonce, block number, and transaction index support structural uniqueness and sequencing evidence only. This does not prove FPG replay prevention without retry/reconciliation event data.

In [ ]:
display(eth[['hash','nonce','block_number','transaction_index','receipt_status']].head())
print(profile['identity_and_replay_fields'])

### 8. Regulatory Required Field Mapping v3

In [ ]:
display(mapping[['required_field','legal_control','ethereum_field','availability_class','confidence','evidence']])

### 9. Ethereum Schema Gap Metrics

In [ ]:
gap = json.loads((ROOT/'03_digital_asset/artifacts/candidate_policy_v3/schema_gap_detailed.json').read_text(encoding='utf-8'))
print(gap['metrics'])
availability = mapping['availability_class'].value_counts().rename_axis('availability_class').reset_index(name='count')
availability.plot.bar(x='availability_class', y='count', legend=False, color='#2f6f8f', title='Ethereum Required Field Availability')
plt.xticks(rotation=45, ha='right'); plt.tight_layout()

### 10. Control-Level Coverage

In [ ]:
display(controls[['control','required_field_count','onchain_field_count','internal_dependency_count','external_dependency_count','pre_execution_dependency_count','runtime_readiness']])
controls.set_index('control')[['onchain_field_count','internal_dependency_count','external_dependency_count']].plot.bar(color=['#2f6f8f','#7a6f45','#8b5a5a'], title='Control-level coverage')
plt.xticks(rotation=45, ha='right'); plt.tight_layout()

### 11. Synthetic v3 Provenance

Synthetic v3 uses real Ethereum distributions for value, timestamps, address concentration, transaction hash, and status; policy fields remain synthetic or derived.

In [ ]:
meta = pd.DataFrame(json.loads((ROOT/'03_digital_asset/data/processed/synthetic_regulated_transfer_v3_metadata.json').read_text(encoding='utf-8')))
display(meta)
display(synthetic.head())

### 12. Policy Removal Sensitivity

These are synthetic policy scenario outcomes, not observed market fraud/failure rates.

In [ ]:
display(sensitivity)
sensitivity.set_index('scenario')[['pass_count','block_count','review_count']].plot.bar(color=['#2f6f8f','#7a6f45','#8b5a5a'], title='Policy Removal - PASS/BLOCK/REVIEW')
plt.xticks(rotation=45, ha='right'); plt.tight_layout()

### 13. False Allow Sensitivity

In [ ]:
sensitivity.set_index('scenario')['false_allow'].plot.bar(color='#a84c3d', title='Policy Removal - False Allow')
plt.xticks(rotation=45, ha='right'); plt.tight_layout()

## Takeaways

H1 is supported by Ethereum schema evidence because identity, KYC, and counterparty VASP status remain non-chain dependencies. H2 is supported by regulatory structure. H3 and H4 are supported by synthetic policy experiments. H5 is structural evidence only, not statistical retry/reconciliation validation.